In [ ]:
!pip install sentence-transformers chromadb pyyaml -q
print("✅ Зависимости установлены")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
import json
import os
from pathlib import Path

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
import gc
from tqdm import tqdm

In [ ]:
CHUNKS_PATH = '/content/all_chunks.json'
VECTOR_DB_PATH = '/content/chromadb'

# Выбор модели (light, balanced, high_quality)
MODEL_NAME = 'light'  # ⚡ light - быстрая, balanced - качественная, high_quality - лучшая

In [ ]:
os.makedirs(VECTOR_DB_PATH, exist_ok=True)

# Загружаем чанки
print(f"\n📂 Загрузка чанков из: {CHUNKS_PATH}")

if not os.path.exists(CHUNKS_PATH):
    print("❌ ОШИБКА: Файл не найден!")
    print("   Проверьте путь CHUNKS_PATH")
else:
    with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    chunks = data.get('chunks', [])
    print(f"✅ Загружено чанков: {len(chunks)}")


📂 Загрузка чанков из: /content/all_chunks.json
✅ Загружено чанков: 110


In [ ]:
#Конфигурация моделей
MODEL_CONFIGS = {
    'light': {
        'repo': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        'batch_size': 4,
        'description': '⚡ Легкая модель (~120MB) - быстрая, для CPU'
    },
    'balanced': {
        'repo': 'intfloat/multilingual-e5-base',
        'batch_size': 8,
        'description': '⚖️ Средняя модель (~1.1GB) - хорошее качество'
    },
    'high_quality': {
        'repo': 'intfloat/multilingual-e5-large',
        'batch_size': 16,
        'description': '🏆 Лучшая модель (~2.2GB) - максимальное качество'
    }
}

# Проверка выбора модели
if MODEL_NAME not in MODEL_CONFIGS:
    print(f"❌ Неизвестная модель: {MODEL_NAME}")
    print(f"   Доступные: {list(MODEL_CONFIGS.keys())}")
    MODEL_NAME = 'light'

config = MODEL_CONFIGS[MODEL_NAME]
BATCH_SIZE = config['batch_size']

print(f"\n🤖 Загрузка модели: {config['repo']}")
print(f"   {config['description']}")

model = SentenceTransformer(config['repo'])
print("✅ Модель загружена!")


🤖 Загрузка модели: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
   ⚡ Легкая модель (~120MB) - быстрая, для CPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Модель загружена!


In [8]:
print(f"\n🗄️ Инициализация ChromaDB...")
print(f"   Путь: {VECTOR_DB_PATH}")

chroma_client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
collection = chroma_client.get_or_create_collection(name="clinical_protocols")
print(f"✅ ChromaDB инициализирована!")


🗄️ Инициализация ChromaDB...
   Путь: /content/drive/MyDrive/ClinAI/data/chromadb
✅ ChromaDB инициализирована!


In [9]:
print(f"\n🚀 Начало векторизации {len(chunks)} чанков...")
print(f"   Размер батча: {BATCH_SIZE}")
print(f"   Всего батчей: {len(chunks) // BATCH_SIZE + 1}")

for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Векторизация"):
    batch = chunks[i:i + BATCH_SIZE]

    # Получаем тексты из чанков
    texts = [c['text'] for c in batch]

    # Создаём эмбеддинги
    embeddings = model.encode(
        texts,
        show_progress_bar=False,
        normalize_embeddings=True,
        batch_size=BATCH_SIZE
    )

    # Подготавливаем данные для ChromaDB
    ids = [f"chunk_{i + j}" for j in range(len(batch))]
    metadatas = [c.get('metadata', {}) for c in batch]

    # Добавляем в коллекцию
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        metadatas=metadatas,
        documents=texts
    )

    # Очищаем память
    gc.collect()


🚀 Начало векторизации 110 чанков...
   Размер батча: 4
   Всего батчей: 28


Векторизация: 100%|██████████| 28/28 [00:27<00:00,  1.04it/s]


In [10]:
print("\n" + "=" * 60)
print("🎉 ВЕКТОРНАЯ БАЗА ДАННЫХ УСПЕШНО СОЗДАНА!")
print("=" * 60)
print(f"📍 Путь: {VECTOR_DB_PATH}")
print(f"📊 Всего документов: {collection.count()}")
print(f"🧠 Модель: {MODEL_CONFIGS[MODEL_NAME]['repo']}")


🎉 ВЕКТОРНАЯ БАЗА ДАННЫХ УСПЕШНО СОЗДАНА!
📍 Путь: /content/drive/MyDrive/ClinAI/data/chromadb
📊 Всего документов: 110
🧠 Модель: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [12]:
print("\n" + "-" * 60)
print("🔍 Тестирование поиска...")

test_query = "Пртериальная гипертензия"
query_embedding = model.encode([test_query], normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print(f"\nЗапрос: '{test_query}'")
print(f"Найдено результатов: {len(results['documents'][0])}")

for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"\n--- Результат {i+1} ---")
    print(f"Текст: {doc[:150]}...")
    if meta:
        print(f"Метаданные: {meta}")


------------------------------------------------------------
🔍 Тестирование поиска...

Запрос: 'Пртериальная гипертензия'
Найдено результатов: 3

--- Результат 1 ---
Текст: **Гипертонический криз:** см. раздел эссенциальной гипертензии...
Метаданные: {'section_type': 'treatment', 'diagnosis': 'Артериальная гипертензия с преимущественным поражением сердца и почек с застойной сердечной недостаточностью', 'icd10_code': 'I13.0', 'subsection': 'Купирование приступа / Острая фаза', 'tags': ['кардиология', 'нефрология']}

--- Результат 2 ---
Текст: **Диагноз:** Реноваскулярная гипертензия (фибро-мускулярная дисплазия, неспецифический аортоартериит)  
**Код МКБ-10:** I15.0  
**Протокол:** Клиничес...
Метаданные: {'icd10_code': 'I15.0', 'diagnosis': 'Реноваскулярная гипертензия (фибро-мускулярная дисплазия, неспецифический аортоартериит)', 'tags': ['кардиология'], 'section_type': 'general', 'subsection': '📋 Общая информация'}

--- Результат 3 ---
Текст: **Диагноз:** Артериальная гипертензия с 